In [1]:
import os
import argparse
import torch
from datasets import MyDataSet
# from vit_model import VisionTransformer
from Residual import Student_fc as Student
from  MLP import MLP


import collections
import math
import shutil
import pandas as pd
import numpy as np
import torchvision
from torch import nn
from torch.utils.data import Dataset
from torch.nn import functional as F
from d2l import torch as d2l
from PIL import Image

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
batch_size = 128
epochs = 101
lr = 0.01    #学习率大小
train_path = r"E:\北邮2024\本科毕设\Model\IndooLocation\test"
test_path =  r"E:\北邮2024\本科毕设\Model\IndooLocation\test"
if os.path.exists("./weights") is False:
    os.makedirs("./weights")

In [3]:
# 实例化训练数据集
train_dataset = MyDataSet(folder_path="test")

fold path is  test
origin_len is 64000
输入数据总维度为  (2000, 1, 32, 3)
标签总维度为  (2000, 1, 32)


In [4]:
# 实例化测试数据集
val_dataset = MyDataSet(folder_path = test_path)

fold path is  E:\北邮2024\本科毕设\Model\IndooLocation\test
origin_len is 64000
输入数据总维度为  (2000, 1, 32, 3)
标签总维度为  (2000, 1, 32)


In [5]:
nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])  # number of workers
print('Using {} dataloader workers every process'.format(nw))

train_loader = torch.utils.data.DataLoader(train_dataset,
                                            batch_size=batch_size,  
                                            # weight_decay=1e-3,
                                            shuffle=True,
                                            pin_memory=True,
                                            num_workers=nw)
val_loader = torch.utils.data.DataLoader(val_dataset,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            pin_memory=True,
                                            num_workers=nw)

Using 8 dataloader workers every process


In [6]:
with open("loss.txt", "w") as f:
    f.write("")
with open("accuracy.txt", "w") as f:
    f.write("")
with open("accuracy_test.txt", "w") as f:
    f.write("")
with open("loss_test.txt", "w") as f:
    f.write("")
with open("train_predictions.csv", "w") as f:
    f.write("")

In [7]:
print("total epochs:",epochs)
from utils import train_one_epoch,test_model
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
model=Student().to(device)
# model = MLP().to(device)
model.train()
optimizer = torch.optim.SGD(model.parameters(), lr = lr)
for epoch in range(epochs):
    # train
    train_loss, train_accuracy = train_one_epoch(model=model,
                                optimizer=optimizer,
                                data_loader=train_loader,
                                device=device,
                                epoch=epoch,
                                save_path="train_predictions.csv")
    optimizer.step()
    with open("loss.txt", "a") as f_loss:
        f_loss.write("loss:{}\n".format(train_loss))
    with open("accuracy.txt", "a") as f_accuracy:
        f_accuracy.write("accuracy:{}\n".format(train_accuracy))
    # validate
    # if (epoch+1) % 5 == 0:
    #     pred, test_loss, labels, test_accuracy = test_model(model=model,
    #                             data_loader=val_loader,
    #                             device=device)
    #     average_loss = sum(test_loss)/len(test_loss)
    #     print("................")
    #     print("验证集结果：")
    #     print(f"平均误差: {average_loss:.3f}")
    #     print(f"准确率: {test_accuracy:.2f}%")
    #     print("................")
    #     with open("accuracy_test.txt", "a") as f:
    #         f.write("accuracy test:{}\n".format(test_accuracy))
    #     with open("loss_test.txt", "a") as f:
    #         f.write("loss test:{}\n".format(average_loss))
        
    if (epoch+1) % 20 == 0:
        print("保存模型")
        # torch.save(model.state_dict(), "./weights/model-{}.pth".format(epoch))
        if not os.path.exists('./model_save'):
            os.makedirs('./model_save')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': train_loss,
            'accuracy': train_accuracy
            }, "./model_save/model-{}.pth".format(epoch))
print("训练完成")

total epochs: 101
 
[train epoch 1] 平均误差:0.051, 平均绝对误差:0.096
 
[train epoch 2] 平均误差:0.021, 平均绝对误差:0.069
 
[train epoch 3] 平均误差:0.016, 平均绝对误差:0.060
 
[train epoch 4] 平均误差:0.010, 平均绝对误差:0.049
 
[train epoch 5] 平均误差:0.009, 平均绝对误差:0.045
 
[train epoch 6] 平均误差:0.007, 平均绝对误差:0.039
 
[train epoch 7] 平均误差:0.006, 平均绝对误差:0.037
 
[train epoch 8] 平均误差:0.007, 平均绝对误差:0.039
 
[train epoch 9] 平均误差:0.006, 平均绝对误差:0.037
 
[train epoch 10] 平均误差:0.005, 平均绝对误差:0.036
 
[train epoch 11] 平均误差:0.005, 平均绝对误差:0.034
 
[train epoch 12] 平均误差:0.004, 平均绝对误差:0.036
 
[train epoch 13] 平均误差:0.005, 平均绝对误差:0.034
 
[train epoch 14] 平均误差:0.005, 平均绝对误差:0.038
 
[train epoch 15] 平均误差:0.004, 平均绝对误差:0.033
 
[train epoch 16] 平均误差:0.004, 平均绝对误差:0.034
 
[train epoch 17] 平均误差:0.004, 平均绝对误差:0.033
 
[train epoch 18] 平均误差:0.004, 平均绝对误差:0.031
 
[train epoch 19] 平均误差:0.004, 平均绝对误差:0.033
 
[train epoch 20] 平均误差:0.003, 平均绝对误差:0.032
保存模型
 
[train epoch 21] 平均误差:0.003, 平均绝对误差:0.031
 
[train epoch 22] 平均误差:0.003, 平均绝对误差:0.032
 
[train epoch 23]

KeyboardInterrupt: 

In [ ]:
pred, test_loss, labels, test_accuracy = test_model(model=model,
                                data_loader=val_loader,
                                device=device)
average_loss = sum(test_loss)/len(test_loss)
print("................")
print("验证集结果：")
print(f"平均误差: {average_loss:.3f}")
print(f"准确率: {test_accuracy:.2f}%")
print("................")

In [ ]:
# from torchinfo import summary

# model = Student()
# summary(model, input_size=(1, 1, 256, 8), col_names=["input_size", "output_size", "num_params", "kernel_size"])